# PhasemeterClient v2 — External AWG frequency loop + wrapped/unwrapped Δφ
This notebook demonstrates:
- connecting/configuring
- measuring wrapped/unwrapped Δφ
- **external AWG-controlled sweep** by passing frequencies in a loop
- saving results to CSV

In [ ]:
from phasemeter_client_v2 import PhasemeterClient
import numpy as np

TARGET = '[fe80::7269:79ff:feb7:d15%6]'  # <-- replace
client = PhasemeterClient(target=TARGET, pll_bandwidth='1kHz', poll_sec=0.05)
client.load(f0_hz=21e6)


## External AWG loop pattern
You said you already have AWG control code. The pattern below is what you need:
- you set the AWG frequency
- then yield that frequency into the phasemeter sweep


In [ ]:
# Example frequency list
freqs = np.linspace(19e6, 23e6, 21)

# Replace this with your real AWG driver
def awg_set_frequency(f_hz):
    # e.g., awg.write(f'FREQ {f_hz}')
    pass

def freq_generator():
    for f in freqs:
        awg_set_frequency(float(f))
        yield float(f)

rows_wrapped = client.sweep_phase_vs_frequency_external(
    freq_generator(),
    settle_s=0.2,
    samples_per_point=5,
    mode='wrapped',
    set_pll_to_frequency=True,
    reacquire_each_step=True
)
client.print_sweep(rows_wrapped)
client.plot_sweep(rows_wrapped, y_key='delta_phi_deg', title='Wrapped Δφ vs Frequency')


## Unwrapped option
Set `mode='unwrapped'` (or `mode='both'`) to record unwrapped Δφ.

In [ ]:
rows_both = client.sweep_phase_vs_frequency_external(
    freq_generator(),
    settle_s=0.2,
    samples_per_point=5,
    mode='both',
    reset_unwrap=True
)
client.plot_sweep(rows_both, y_key='delta_phi_unwrapped_deg', title='Unwrapped Δφ vs Frequency')


## Save to CSV

In [ ]:
csv_path = client.save_rows_to_csv(
    rows_both,
    'phase_vs_frequency.csv',
    metadata={
        'target': client.target,
        'pll_bandwidth': client.pll_bandwidth,
        'impedance': client.impedance,
        'coupling': client.coupling,
        'range': client.input_range,
        'phase_units': client.phase_units
    }
)
print('Saved:', csv_path)


In [ ]:
client.close()
